In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%sql
use catalog lendingclub;
create schema if not exists silver_cleaned;
use schema silver_cleaned;
select current_catalog(), current_schema();

In [0]:
loans_raw_df = spark.read.table('bronze.loans')

In [0]:
#ingested date

loan_ingested_df = loans_raw_df.withColumn('ingested_date', current_timestamp())

In [0]:
#check if loan_amount is null

loan_ingested_df.filter('loan_amount is null').count()

In [0]:
loan_filtered_df = loan_ingested_df.filter('loan_amount is not null')

In [0]:
loan_filtered_df.filter('funded_amount is null').count()

In [0]:
#Convert loan term into years 

loan_years_df = loan_filtered_df.withColumn('loan_years',\
                                            (regexp_replace(col('loan_term'),r'\D','')/12).cast('integer'))

In [0]:
display(loan_years_df.head(5))

In [0]:
#clearn loan status/loan purpose - remove unwanted purpose

display(loan_years_df.select('loan_status').distinct())

In [0]:
#loan_years_df.select('loan_purpose').distinct()
display(loan_years_df.groupby('loan_purpose').count().sort(desc('count')))

In [0]:
#update remaining loan purpsoe into others

loan_purpose_lookup = ['debt_consolidation','credit_card','home_improvement','other','major_purchase','medical','small_business','car','vacation','moving','house''wedding','renewable_energy','educational']

loan_modified_df = loan_years_df.withColumn('loan_purpose',\
                        when(col('loan_purpose').isin(loan_purpose_lookup),col('loan_purpose'))\
                        .otherwise('other'))

In [0]:
display(loan_modified_df.groupby('loan_purpose').count().sort(desc('count')))

In [0]:
dbutils.fs.rm("/Volumes/lendingclub/storagelocation/cleaned/Loans/", recurse=True)

In [0]:
loan_modified_df.write.format('delta').mode('overwrite').save('/Volumes/lendingclub/storagelocation/cleaned/Loans/')

In [0]:
%sql
create or replace table silver_cleaned.loans
as
select * 
from delta.`/Volumes/lendingclub/storagelocation/cleaned/Loans/`